# GOT-OCR-2.0: handwritten image -> text

Runs `stepfun-ai/GOT-OCR-2.0-hf` (pretrained, no fine-tuning) over every image in a Drive
folder. Structured the same way as `TROCR_testing.ipynb`: preprocessing kept, no grid/cell
splitting into per-field boxes.

Unlike TrOCR, GOT-OCR-2.0 is not limited to a single text line per input -- it can read a
whole multi-line page directly. So this notebook feeds the **preprocessed page as a whole**
straight to GOT, with no line-segmentation step.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled (Runtime -> Change runtime type -> GPU). GOT-OCR will be slow on CPU.")

In [ ]:
!pip install -q -U transformers accelerate sentencepiece opencv-python-headless

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

In [ ]:
DIRECTORY_PATH = "/content/drive/MyDrive/Meenakshi/Dataset/handwriteen/"

image_files = []
for root, _, files in os.walk(DIRECTORY_PATH):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_files.append(os.path.join(root, file))

image_files.sort()
print(f"Found {len(image_files)} image files in {DIRECTORY_PATH}")
if len(image_files) > 5:
    print("First 5 image files:", image_files[:5])
else:
    print("Image files:", image_files)

In [ ]:
if image_files:
    current_image_path = image_files[0]
    image = Image.open(current_image_path).convert("RGB")

    print("Image size:", image.size)

    plt.figure(figsize=(14, 7))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Original handwritten image: {os.path.basename(current_image_path)}")
    plt.show()
else:
    print("No image files found in the specified directory.")

## Preprocessing

Upscale + contrast-boost + denoise, same as the TrOCR notebook: small/faint handwriting
needs the resolution and contrast boost to give the model a chance. Runs on the **whole
page** (no per-cell cropping).

In [ ]:
def preprocess_handwriting(image, scale=3):
    """
    Enhance a handwriting image for OCR: grayscale, upscale, local contrast boost,
    light denoise. Returns a grayscale numpy array.
    """
    img = np.array(image.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    upscaled = cv2.resize(
        gray,
        None,
        fx=scale,
        fy=scale,
        interpolation=cv2.INTER_CUBIC
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )
    enhanced = clahe.apply(upscaled)

    denoised = cv2.fastNlMeansDenoising(
        enhanced,
        None,
        h=5,
        templateWindowSize=7,
        searchWindowSize=21
    )

    return denoised

In [ ]:
PREPROCESS_SCALE = 3
enhanced = preprocess_handwriting(image, scale=PREPROCESS_SCALE)

plt.figure(figsize=(14, 7))
plt.imshow(enhanced, cmap="gray")
plt.axis("off")
plt.title("Enhanced (this is what gets fed to OCR)")
plt.show()

## Load GOT-OCR-2.0

In [ ]:
MODEL_ID = "stepfun-ai/GOT-OCR-2.0-hf"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading model...")

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

print("Model loaded")

In [ ]:
def got_ocr(image, max_new_tokens=512, crop_to_patches=False, max_patches=3):
    """
    Run GOT-OCR 2.0 on an image (grayscale numpy array or PIL image) and return
    (decoded_text, confidence). Confidence is the mean per-token generation
    probability (exp of mean log-prob of the chosen token at each step),
    a simple proxy for how sure the model was about its own output.
    """
    if isinstance(image, np.ndarray):
        if len(image.shape) == 2:
            image = Image.fromarray(image)
        else:
            image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    image = image.convert("RGB")

    processor_kwargs = {"return_tensors": "pt"}

    if crop_to_patches:
        processor_kwargs["crop_to_patches"] = True
        processor_kwargs["max_patches"] = max_patches

    inputs = processor(image, **processor_kwargs)

    inputs = {
        k: v.to(model.device) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            tokenizer=processor.tokenizer,
            stop_strings="<|im_end|>",
            max_new_tokens=max_new_tokens,
            output_scores=True,
            return_dict_in_generate=True,
        )

    generated_ids = outputs.sequences
    input_length = inputs["input_ids"].shape[1]
    new_token_ids = generated_ids[0][input_length:]

    generated_text = processor.decode(
        new_token_ids,
        skip_special_tokens=True
    )

    # Mean token probability over the generated (non-special) tokens.
    token_probs = []
    for step_scores, token_id in zip(outputs.scores, new_token_ids):
        if token_id in processor.tokenizer.all_special_ids:
            continue
        step_probs = torch.softmax(step_scores[0], dim=-1)
        token_probs.append(step_probs[token_id].item())

    confidence = float(np.mean(token_probs)) if token_probs else float("nan")

    return generated_text.strip(), confidence


def got_read_image(image, scale=PREPROCESS_SCALE, max_new_tokens=512, crop_to_patches=False, max_patches=3):
    """
    Preprocess + OCR one page image with GOT-OCR-2.0 (no cell/grid/line splitting).
    Returns (text, confidence, enhanced_array).
    """
    enhanced = preprocess_handwriting(image, scale=scale)
    text, confidence = got_ocr(
        enhanced,
        max_new_tokens=max_new_tokens,
        crop_to_patches=crop_to_patches,
        max_patches=max_patches
    )
    return text, confidence, enhanced

## Try it on one image

If the page is large/dense, set `crop_to_patches=True` -- GOT-OCR-2.0 splits the image into
overlapping patches (up to `max_patches`) and stitches the OCR output, which helps on pages
with a lot of small text.

In [ ]:
preview_image = Image.open(image_files[0]).convert("RGB")
preview_text, preview_confidence, preview_enhanced = got_read_image(preview_image)

print(f"{os.path.basename(image_files[0])}\n")
print(f"Confidence: {preview_confidence:.4f}")
print("OCR text:")
print(preview_text)

## Run GOT-OCR-2.0 on every image and save results

In [ ]:
got_results = {}

for img_path in image_files:
    print(f"Processing image: {os.path.basename(img_path)}")
    try:
        image = Image.open(img_path).convert("RGB")
        text, confidence, _ = got_read_image(image)
        got_results[img_path] = {"text": text, "confidence": confidence}
        print(f"-> done (confidence: {confidence:.4f})")
    except Exception as e:
        print(f"Error processing {os.path.basename(img_path)}: {e}")

processed_data = [
    {"image_name": os.path.basename(p), "ocr_text": r["text"], "confidence": r["confidence"]}
    for p, r in got_results.items()
]
final_df = pd.DataFrame(processed_data)
print(f"\nProcessed {len(final_df)} image(s).")

OUTPUT_DIR = "/content/drive/MyDrive/Meenakshi/Dataset/ocr_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_path = os.path.join(OUTPUT_DIR, "got_results.csv")
json_path = os.path.join(OUTPUT_DIR, "got_results.json")

final_df.to_csv(csv_path, index=False, encoding="utf-8")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        {os.path.basename(p): r for p, r in got_results.items()},
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved CSV:", csv_path)
print("Saved JSON:", json_path)

In [ ]:
final_df